In [0]:
%sql
USE CATALOG workspace;

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS toy_store

In [0]:
%sql
USE DATABASE toy_store

In [0]:
%sql
DROP TABLE IF EXISTS workspace.toy_store.orders

In [0]:
  %fs ls /Volumes/workspace/bronze/raw_sources/toy_store_e-commerce/

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.bronze.orders AS
SELECT *
FROM read_files(
  'dbfs:/Volumes/workspace/bronze/raw_sources/toy_store_e-commerce/orders.csv',
  format => 'csv',
  header => true,
  inferSchema => false
)


In [0]:
%sql
SELECT * FROM workspace.bronze.orders

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.bronze.order_item_refunds AS
SELECT *
FROM read_files(
  'dbfs:/Volumes/workspace/bronze/raw_sources/toy_store_e-commerce/order_item_refunds.csv',
  format => 'csv',
  header => true,
  inferSchema => false
)

In [0]:
%sql
SELECT * FROM workspace.bronze.order_item_refunds

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.bronze.order_items AS
SELECT *
FROM read_files(
  'dbfs:/Volumes/workspace/bronze/raw_sources/toy_store_e-commerce/order_items.csv',
  format => 'csv',
  header => true,
  inferSchema => false
)

In [0]:
%sql
SELECT * FROM workspace.bronze.order_items

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.bronze.products AS
SELECT *
FROM read_files(
  'dbfs:/Volumes/workspace/bronze/raw_sources/toy_store_e-commerce/products.csv',
  format => 'csv',
  header => true,
  inferSchema => false
)

In [0]:
%sql
SELECT * FROM workspace.bronze.products

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.bronze.website_pageviews AS
SELECT *
FROM read_files(
  'dbfs:/Volumes/workspace/bronze/raw_sources/toy_store_e-commerce/website_pageviews.csv',
  format => 'csv',
  header => true,
  inferSchema => false
)

In [0]:
%sql
SELECT * FROM workspace.bronze.website_pageviews

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.bronze.website_sessions AS
SELECT *
FROM read_files(
  'dbfs:/Volumes/workspace/bronze/raw_sources/toy_store_e-commerce/website_sessions.csv',
  format => 'csv',
  header => true,
  inferSchema => false
)

In [0]:
%sql
SELECT * FROM workspace.bronze.website_sessions

In [0]:
%sql
-- SILVER: orders item refunds
CREATE OR REPLACE TABLE workspace.silver.order_item_refunds AS
SELECT
  order_item_refund_id,
  to_timestamp(created_at)                    AS refund_date,
  CAST(order_item_id AS INT)                  AS order_item_id,
  order_id,
  CAST(refund_amount_usd AS DECIMAL(10,2))    AS refund_amount_usd
FROM bronze.order_item_refunds

In [0]:
%sql
SELECT * FROM silver.order_item_refunds

In [0]:
%sql
-- SILVER: order items

CREATE OR REPLACE TABLE workspace.silver.order_items AS
SELECT
  order_item_id,
  to_timestamp(created_at)                    AS order_date,
  CAST(order_id AS INT)                       AS order_id,
  CAST(product_id AS INT)                     AS product_id,
  CAST(is_primary_item AS INT)                AS is_primary_item,
  CAST(price_usd AS DECIMAL(10,2))            AS price_usd,
  CAST(cogs_usd AS DECIMAL(10,2))             AS cogs_usd
FROM bronze.order_items


In [0]:
%sql
select * from silver.order_items

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.orders AS
SELECT
  order_id,
  to_timestamp(created_at)            AS order_date,
  CAST(website_session_id AS INT)     AS website_session_id,
  CAST(user_id AS INT)                AS user_id,
  CAST(primary_product_id AS INT)     AS primary_product_id,
  CAST(items_purchased AS INT)        AS no_of_orders,
  CAST(price_usd AS DECIMAL(10,2))    AS price,
  CAST(cogs_usd AS DECIMAL(10,2))     AS costs
FROM bronze.orders

In [0]:
%sql
DESCRIBE TABLE silver.orders;

In [0]:
%sql
select * from bronze.orders

In [0]:
%sql
select * from bronze.order_items

In [0]:
%sql
select count(*) from bronze.orders

In [0]:
%sql
SELECT 
*
FROM bronze.products

In [0]:
%sql
-- SILVER: products table
CREATE OR REPLACE TABLE workspace.silver.products AS
SELECT
  CAST(product_id AS INT)   AS product_id,
  to_timestamp(created_at)  AS product_date,
  product_name              AS product_name
FROM bronze.products

In [0]:
%sql
DESCRIBE TABLE silver.products

In [0]:
%sql
SELECT * FROM silver.products

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.website_sessions AS
SELECT
  CAST(website_session_id AS INT) AS session_id,
  to_timestamp(created_at)        AS session_date,
  CAST(user_id AS INT)            AS user_id,
  CAST(is_repeat_session AS INT)  AS is_repeat_session,
  CAST(utm_source AS string)      AS utm_source,
  CAST(utm_campaign AS string)    AS utm_campaign,
  CAST(utm_content AS string)     AS utm_content,
  CAST(device_type AS string)     AS device_type,
  CAST(http_referer AS string)    AS http_referer
FROM bronze.website_sessions

In [0]:
%sql
DESCRIBE TABLE workspace.bronze.website_sessions

In [0]:
%sql
DESCRIBE TABLE silver.website_sessions

In [0]:
%sql
SELECT * FROM bronze.website_pageviews

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.website_pageviews AS
select
  website_pageview_id,
  to_timestamp(created_at)          AS  session_date,
  cast(website_session_id AS INT)   AS  session_id,
case 
  when substring(pageview_url, 2) in ('home', 'lander-1', 'lander-2', 'lander-3', 'lander-4', 'lander-5') then 'landing'
  when substring(pageview_url, 2) in ('billing', 'billing-2') then 'billing'
  else substring(pageview_url, 2) end as page_view_url
from bronze.website_pageviews

In [0]:
%sql
SELECT DISTINCT page_view_url, count(page_view_url) FROM workspace.silver.website_pageviews group by page_view_url

In [0]:
%sql
SELECT 
  'website_pageviews' as table_name, count(*) as total_records from workspace.silver.website_pageviews
UNION ALL
SELECT 
  'products' as table_name, count(*) as total_records from workspace.silver.products
UNION ALL
SELECT 
  'website_sessions' as table_name, count(*) as total_records from workspace.silver.website_sessions
UNION ALL
SELECT 
  'orders' as table_name, count(*) as total_records from workspace.silver.orders
UNION ALL
SELECT 
  'order_item_refunds' as table_name, count(*) as total_records from workspace.silver.order_item_refunds
  UNION
SELECT
  'order_items' as table_name, count(*) as total_records from workspace.silver.order_items

In [0]:
%sql

- total revenue
select sum(price_usd) from silver.order_items

In [0]:
%sql
--total revenue by year
SELECT year(order_date) as year, sum(price_usd) as total_revenue FROM workspace.silver.order_items group by year(order_date)


In [0]:
%sql
--total revenue by year and by product
SELECT 
    YEAR(oi.order_date) AS year,
    SUM(oi.price_usd) AS total_revenue,
    SUM(CASE WHEN pr.product_name = 'The Original Mr. Fuzzy' THEN oi.price_usd ELSE 0 END) AS mr_fuzzy_revenue,
    SUM(CASE WHEN pr.product_name = 'The Forever Love Bear' THEN oi.price_usd ELSE 0 END) AS love_bear_revenue,
    SUM(CASE WHEN pr.product_name = 'The Birthday Sugar Panda' THEN oi.price_usd ELSE 0 END) AS sugar_panda_revenue,
    SUM(CASE WHEN pr.product_name = 'The Hudson River Mini bear' THEN oi.price_usd ELSE 0 END) AS mini_bear_revenue
FROM silver.order_items oi
INNER JOIN silver.products pr 
    ON oi.product_id = pr.product_id
GROUP BY YEAR(oi.order_date)
ORDER BY year;


--workspace.silver.order_items group by year(order_date)

In [0]:
%sql
SELECT 
    date_format(oi.order_date, 'MMM-yyyy') AS year,
    SUM(oi.price_usd) AS total_revenue,
    SUM(CASE WHEN pr.product_name = 'The Original Mr. Fuzzy' THEN oi.price_usd ELSE 0 END) AS mr_fuzzy_revenue,
    SUM(CASE WHEN pr.product_name = 'The Forever Love Bear' THEN oi.price_usd ELSE 0 END) AS love_bear_revenue,
    SUM(CASE WHEN pr.product_name = 'The Birthday Sugar Panda' THEN oi.price_usd ELSE 0 END) AS sugar_panda_revenue,
    SUM(CASE WHEN pr.product_name = 'The Hudson River Mini bear' THEN oi.price_usd ELSE 0 END) AS mini_bear_revenue
FROM silver.order_items oi
INNER JOIN silver.products pr 
    ON oi.product_id = pr.product_id
GROUP BY date_format(oi.order_date, 'MMM-yyyy')
ORDER BY date_format(oi.order_date, 'MMM-yyyy');

In [0]:
%sql
CREATE OR REPLACE TABLE mart_sessions AS
WITH first_page AS (
    SELECT
        website_session_id,
        MIN(created_at) AS first_page_time,
        MIN(pageview_url) KEEP (DENSE_RANK FIRST ORDER BY created_at) AS landing_page
    FROM stg_website_pageviews
    GROUP BY website_session_id
),
session_orders AS (
    SELECT
        website_session_id,
        COUNT(order_id) AS orders,
        SUM(price_usd) AS revenue
    FROM stg_orders
    GROUP BY website_session_id
)
SELECT
    s.website_session_id,
    s.created_at AS session_start,
    s.device_type,
    s.utm_source,
    s.utm_campaign,
    s.utm_content,
    fp.landing_page,
    COALESCE(o.orders, 0) AS orders,
    COALESCE(o.revenue, 0) AS revenue
FROM stg_website_sessions s
LEFT JOIN first_page fp ON s.website_session_id = fp.website_session_id
LEFT JOIN session_orders o ON s.website_session_id = o.website_session_id;


In [0]:
%sql
-- total

In [0]:
%sql
--total revenue by year
SELECT 
    date_format(oi.order_date) AS year,
    SUM(oi.price_usd) AS total_revenue,
    SUM(CASE WHEN pr.product_name = 'The Original Mr. Fuzzy' THEN oi.price_usd ELSE 0 END) AS mr_fuzzy_revenue,
    SUM(CASE WHEN pr.product_name = 'The Forever Love Bear' THEN oi.price_usd ELSE 0 END) AS love_bear_revenue,
    SUM(CASE WHEN pr.product_name = 'The Birthday Sugar Panda' THEN oi.price_usd ELSE 0 END) AS sugar_panda_revenue,
    SUM(CASE WHEN pr.product_name = 'The Hudson River Mini bear' THEN oi.price_usd ELSE 0 END) AS mini_bear_revenue
FROM silver.order_items oi
INNER JOIN silver.products pr 
    ON oi.product_id = pr.product_id
GROUP BY YEAR(oi.order_date)
ORDER BY year;


In [0]:
%sql
select * from silver.products

In [0]:
%sql
-- TOTAL REVENUE

SELECT
  *
FROM 
  workspace

In [0]:
%sql
SELECT * FROM bronze.website_sessions

In [0]:
%sql
SELECT * FROM bronze.orders

In [0]:
%sql
SELECT * FROM bronze.order_item_refunds

In [0]:
%sql
SELECT DISTINCT utm_campaign, utm_content, http_referer
FROM workspace.bronze.website_sessions

In [0]:
%sql
SELECT * FROM workspace.bronze.website_pageviews;

In [0]:
%sql
-- SILVER: 

In [0]:
%sql
select distinct utm_campaign, utm_content, http_referer
from workspace.bronze.website_sessions;

In [0]:
%sql
SELECT * FROM website_sessions;

In [0]:
%sql
SELECT * FROM bronze.website_pageviews;

In [0]:
%sql
SELECT COUNT(*) FROM bronze.website_sessions

In [0]:
%sql
SELECT count(website_session_id) FROM bronze.website_pageviews

In [0]:
%sql
SELECT COUNT(*), COUNT(website_session_id)
FROM bronze.website_pageviews;


In [0]:
%sql
SELECT website_session_id, COUNT(*)
FROM bronze.website_pageviews
GROUP BY website_session_id
HAVING COUNT(*) > 1;


In [0]:
%sql
SELECT COUNT(*)
FROM bronze.website_pageviews
WHERE website_session_id IS NULL OR TRIM(website_session_id) = '';


In [0]:
%sql
SELECT DISTINCT
  website_pageviews.pageview_url,
  count(website_pageviews.pageview_url)
FROM 
  bronze.website_sessions
LEFT JOIN bronze.website_pageviews
ON website_sessions.website_session_id = website_pageviews.website_session_id
GROUP BY website_pageviews.pageview_url
ORDER BY 1


In [0]:
%sql
select count(*)FROM bronze.website_pageviews where pageview_url = '/the-original-mr-fuzzy'

In [0]:
%sql
SELECT 
  website_sessions.website_session_id,
  website_pageviews.website_pageview_id,
  website_pageviews.pageview_url,
  CASE WHEN website_pageviews.pageview_url IN ('/home', '/lander-1','/lander-2','/lander-3','/lander-4','/lander-5') THEN 1 ELSE 0 END AS landing_flag,
  CASE WHEN website_pageviews.pageview_url = '/products' THEN 1 ELSE 0 END AS product_flag,
  CASE WHEN website_pageviews.pageview_url = '/the-original-mr-fuzzy' THEN 1 ELSE 0 END AS the_original_mr_fuzzy,
  CASE WHEN website_pageviews.pageview_url = '/cart' THEN 1 ELSE 0 END AS cart_flag,
  CASE WHEN website_pageviews.pageview_url = '/shipping' THEN 1 ELSE 0 END AS shipping_flag,
  CASE WHEN website_pageviews.pageview_url IN ('/billing', '/billing-2') THEN 1 ELSE 0 END AS billing_flag,
  CASE WHEN website_pageviews.pageview_url = '/thank-you-for-your-order' THEN 1 ELSE 0 END AS thankyou_flag
FROM 
  bronze.website_sessions
left join bronze.website_pageviews
on website_sessions.website_session_id = website_pageviews.website_session_id

In [0]:
%sql
WITH sessions_with_flags AS (
SELECT 
  website_session_id,
  max(landing_flag) as lander1_made_it,
  max(product_flag) as product_made_it,
  max(view_products) as viewed_product_made_it,
  max(cart_flag) as cart_made_it,
  max(shipping_flag) as shipping_made_it,
  max(billing_flag) as billing_made_it,
  max(thankyou_flag) as thankyou_made_it
FROM (
SELECT 
  website_sessions.website_session_id,
  website_pageviews.website_pageview_id,
  website_pageviews.pageview_url,
  CASE WHEN website_pageviews.pageview_url IN ('/home', '/lander-1','/lander-2','/lander-3','/lander-4','/lander-5') THEN 1 ELSE 0 END AS landing_flag,
  CASE WHEN website_pageviews.pageview_url = '/products' THEN 1 ELSE 0 END AS product_flag,
  CASE WHEN website_pageviews.pageview_url LIKE '/the-%' THEN 1 ELSE 0 END AS view_products,
  CASE WHEN website_pageviews.pageview_url = '/cart' THEN 1 ELSE 0 END AS cart_flag,
  CASE WHEN website_pageviews.pageview_url = '/shipping' THEN 1 ELSE 0 END AS shipping_flag,
  CASE WHEN website_pageviews.pageview_url IN ('/billing', '/billing-2') THEN 1 ELSE 0 END AS billing_flag,
  CASE WHEN website_pageviews.pageview_url = '/thank-you-for-your-order' THEN 1 ELSE 0 END AS thankyou_flag
FROM 
  bronze.website_sessions
left join bronze.website_pageviews
on website_sessions.website_session_id = website_pageviews.website_session_id
where
--website_sessions.created_at > '2012-08-05' and
--website_sessions.created_at < '2012-09-05' and
--website_sessions.utm_source = 'gsearch' --and
--website_sessions.utm_source = 'bsearch' --and
--website_sessions.utm_source = 'bsearch'
--website_sessions.utm_campaign = 'nonbrand'
)
GROUP BY website_session_id
)
SELECT 
COUNT(DISTINCT website_session_id) as sessions,
COUNT(DISTINCT CASE WHEN lander1_made_it=1 THEN website_session_id ELSE NULL END) AS to_landingpage,
COUNT(DISTINCT CASE WHEN product_made_it=1 THEN website_session_id ELSE NULL END) AS to_product,
COUNT(DISTINCT CASE WHEN viewed_product_made_it=1 THEN website_session_id ELSE NULL END) AS viewed_product,
COUNT(DISTINCT CASE WHEN cart_made_it=1 THEN website_session_id ELSE NULL END) AS to_cart,
COUNT(DISTINCT CASE WHEN shipping_made_it=1 THEN website_session_id ELSE NULL END) AS to_shipping,
COUNT(DISTINCT CASE WHEN billing_made_it=1 THEN website_session_id ELSE NULL END) AS to_billing,
COUNT(DISTINCT CASE WHEN thankyou_made_it=1 THEN website_session_id ELSE NULL END) AS to_thank_you_page
FROM sessions_with_flags

In [0]:
%sql
SELECT
COUNT(*),
utm_source
FROM bronze.website_sessions
group by utm_source

In [0]:
%sql
    SELECT 
      ws.website_session_id,
      ws.utm_source,
      MAX(CASE WHEN wp.pageview_url IN ('/home', '/lander-1','/lander-2','/lander-3','/lander-4','/lander-5') THEN 1 ELSE 0 END) AS lander1_made_it,
      MAX(CASE WHEN wp.pageview_url = '/products' THEN 1 ELSE 0 END) AS product_made_it,
      MAX(CASE WHEN wp.pageview_url LIKE '/the-%' THEN 1 ELSE 0 END) AS viewed_product_made_it,
      MAX(CASE WHEN wp.pageview_url = '/cart' THEN 1 ELSE 0 END) AS cart_made_it,
      MAX(CASE WHEN wp.pageview_url = '/shipping' THEN 1 ELSE 0 END) AS shipping_made_it,
      MAX(CASE WHEN wp.pageview_url IN ('/billing', '/billing-2') THEN 1 ELSE 0 END) AS billing_made_it,
      MAX(CASE WHEN wp.pageview_url = '/thank-you-for-your-order' THEN 1 ELSE 0 END) AS thankyou_made_it
    FROM bronze.website_sessions ws
    LEFT JOIN bronze.website_pageviews wp
      ON ws.website_session_id = wp.website_session_id
    GROUP BY ws.website_session_id, ws.utm_source

In [0]:
%sql
WITH sessions_with_flags AS (

SELECT 
  ws.website_session_id,
  ws.utm_source,
 -- wp.website_pageview_id,
  wp.pageview_url,
  MAX(CASE WHEN wp.pageview_url IN ('/home', '/lander-1','/lander-2','/lander-3','/lander-4','/lander-5') THEN 1 ELSE 0 END) AS landing_flag,
  MAX(CASE WHEN wp.pageview_url = '/products' THEN 1 ELSE 0 END) AS product_flag,
  MAX(CASE WHEN wp.pageview_url LIKE '/the-%' THEN 1 ELSE 0 END) AS view_products,
  MAX(CASE WHEN wp.pageview_url = '/cart' THEN 1 ELSE 0 END) AS cart_flag,
  MAX(CASE WHEN wp.pageview_url = '/shipping' THEN 1 ELSE 0 END) AS shipping_flag,
  MAX(CASE WHEN wp.pageview_url IN ('/billing', '/billing-2') THEN 1 ELSE 0 END) AS billing_flag,
  MAX(CASE WHEN wp.pageview_url = '/thank-you-for-your-order' THEN 1 ELSE 0 END) AS thankyou_flag
FROM 
  bronze.website_sessions ws
left join bronze.website_pageviews wp
on ws.website_session_id = wp.website_session_id
GROUP BY ws.website_session_id, ws.utm_source, wp.pageview_url
)
SELECT
utm_source,
COUNT(DISTINCT website_session_id) as sessions,
COUNT(DISTINCT CASE WHEN landing_flag=1 THEN website_session_id ELSE NULL END) AS to_landingpage,
COUNT(DISTINCT CASE WHEN product_flag=1 THEN website_session_id ELSE NULL END) AS to_product,
COUNT(DISTINCT CASE WHEN view_products=1 THEN website_session_id ELSE NULL END) AS viewed_product,
COUNT(DISTINCT CASE WHEN cart_flag=1 THEN website_session_id ELSE NULL END) AS to_cart,
COUNT(DISTINCT CASE WHEN shipping_flag=1 THEN website_session_id ELSE NULL END) AS to_shipping,
COUNT(DISTINCT CASE WHEN billing_flag=1 THEN website_session_id ELSE NULL END) AS to_billing,
COUNT(DISTINCT CASE WHEN thankyou_flag=1 THEN website_session_id ELSE NULL END) AS to_thank_you_page
FROM sessions_with_flags
GROUP BY utm_source

In [0]:
%sql
WITH sessions_with_flags AS (
SELECT 
  website_session_id,
  max(landing_flag) as lander1_made_it,
  max(product_flag) as product_made_it,
  max(the_original_mr_fuzzy) as the_original_mr_fuzzy_made_it,
  max(cart_flag) as cart_made_it,
  max(shipping_flag) as shipping_made_it,
  max(billing_flag) as billing_made_it,
  max(thankyou_flag) as thankyou_made_it
FROM (
SELECT 
  website_sessions.website_session_id,
  website_pageviews.website_pageview_id,
  website_pageviews.pageview_url,
  CASE WHEN website_pageviews.pageview_url IN ('/home', '/lander-1','/lander-2','/lander-3','/lander-4','/lander-5') THEN 1 ELSE 0 END AS landing_flag,
  CASE WHEN website_pageviews.pageview_url = '/products' THEN 1 ELSE 0 END AS product_flag,
  CASE WHEN website_pageviews.pageview_url = '/the-original-mr-fuzzy' THEN 1 ELSE 0 END AS the_original_mr_fuzzy,
  CASE WHEN website_pageviews.pageview_url = '/cart' THEN 1 ELSE 0 END AS cart_flag,
  CASE WHEN website_pageviews.pageview_url = '/shipping' THEN 1 ELSE 0 END AS shipping_flag,
  CASE WHEN website_pageviews.pageview_url IN ('/billing', '/billing-2') THEN 1 ELSE 0 END AS billing_flag,
  CASE WHEN website_pageviews.pageview_url = '/thank-you-for-your-order' THEN 1 ELSE 0 END AS thankyou_flag
FROM 
  bronze.website_sessions
left join bronze.website_pageviews
on website_sessions.website_session_id = website_pageviews.website_session_id
--where
--website_sessions.created_at > '2012-08-05' and
--website_sessions.created_at < '2012-09-05' and
--website_sessions.utm_source = 'bsearch' and
--website_sessions.utm_campaign = 'nonbrand'
)
GROUP BY website_session_id
)
SELECT 
COUNT(DISTINCT website_session_id) as sessions,
concat(cast(round(100 * COUNT(DISTINCT CASE WHEN product_made_it=1 THEN website_session_id ELSE NULL END)/COUNT(DISTINCT website_session_id), 2) AS STRING), '%') as clicked_to_product,
concat(cast(round(100 * COUNT(DISTINCT CASE WHEN the_original_mr_fuzzy_made_it=1 THEN website_session_id ELSE NULL END)/COUNT(DISTINCT CASE WHEN product_made_it=1 THEN website_session_id ELSE NULL END), 2) AS STRING), '%') as clicked_to_the_original_mr_fuzzy,
concat(cast(round(100 * COUNT(DISTINCT CASE WHEN cart_made_it=1 THEN website_session_id ELSE NULL END)/COUNT(DISTINCT CASE WHEN the_original_mr_fuzzy_made_it=1 THEN website_session_id ELSE NULL END), 2) AS STRING), '%') as clieked_to_cart,
concat(cast(round(100 * COUNT(DISTINCT CASE WHEN shipping_made_it=1 THEN website_session_id ELSE NULL END)/COUNT(DISTINCT CASE WHEN cart_made_it=1 THEN website_session_id ELSE NULL END), 2) AS STRING), '%') as clicked_to_shipping,
concat(cast(round(100 * COUNT(DISTINCT CASE WHEN billing_made_it=1 THEN website_session_id ELSE NULL END)/COUNT(DISTINCT CASE WHEN shipping_made_it=1 THEN website_session_id ELSE NULL END), 2) AS STRING), '%') as clicked_to_billing,
concat(cast(round(100 * COUNT(DISTINCT CASE WHEN thankyou_made_it=1 THEN website_session_id ELSE NULL END)/COUNT(DISTINCT CASE WHEN billing_made_it=1 THEN website_session_id ELSE NULL END), 2) AS STRING), '%') as clicked_to_thank_you
FROM sessions_with_flags

In [0]:
%sql
SELECT DISTINCT
  website_pageviews.pageview_url
FROM 
  bronze.website_sessions
LEFT JOIN bronze.website_pageviews
ON website_sessions.website_session_id = website_pageviews.website_session_id
WHERE
website_sessions.created_at > '2012-08-05' and
website_sessions.created_at < '2012-09-05' and
website_sessions.utm_source = 'gsearch' and
website_sessions.utm_campaign = 'nonbrand'

In [0]:
%sql
SELECT 
  website_sessions.website_session_id,
  website_pageviews.website_pageview_id,
  website_pageviews.pageview_url,
  CASE WHEN website_pageviews.pageview_url = '/lander-1' THEN 1 ELSE 0 END AS lander1_flag,
  CASE WHEN website_pageviews.pageview_url = '/products' THEN 1 ELSE 0 END AS product_flag,
  CASE WHEN website_pageviews.pageview_url = '/the-original-mr-fuzzy' THEN 1 ELSE 0 END AS the_original_mr_fuzzy,
  CASE WHEN website_pageviews.pageview_url = '/cart' THEN 1 ELSE 0 END AS cart_flag,
  CASE WHEN website_pageviews.pageview_url = '/shipping' THEN 1 ELSE 0 END AS shipping_flag,
  CASE WHEN website_pageviews.pageview_url = '/billing' THEN 1 ELSE 0 END AS billing_flag,
  CASE WHEN website_pageviews.pageview_url = '/thank-you-for-your-order' THEN 1 ELSE 0 END AS thankyou_flag
FROM 
  bronze.website_sessions
left join bronze.website_pageviews
on website_sessions.website_session_id = website_pageviews.website_session_id
where
website_sessions.created_at > '2012-08-05' and
website_sessions.created_at < '2012-09-05' and
website_sessions.utm_source = 'gsearch' and
website_sessions.utm_campaign = 'nonbrand'

In [0]:
%sql
WITH sessions_with_flags AS (
SELECT 
  website_session_id,
  max(lander1_flag) as lander1_made_it,
  max(product_flag) as product_made_it,
  max(the_original_mr_fuzzy) as the_original_mr_fuzzy_made_it,
  max(cart_flag) as cart_made_it,
  max(shipping_flag) as shipping_made_it,
  max(billing_flag) as billing_made_it,
  max(thankyou_flag) as thankyou_made_it
FROM (
SELECT 
  website_sessions.website_session_id,
  website_pageviews.website_pageview_id,
  website_pageviews.pageview_url,
  CASE WHEN website_pageviews.pageview_url = '/lander-1' THEN 1 ELSE 0 END AS lander1_flag,
  CASE WHEN website_pageviews.pageview_url = '/products' THEN 1 ELSE 0 END AS product_flag,
  CASE WHEN website_pageviews.pageview_url = '/the-original-mr-fuzzy' THEN 1 ELSE 0 END AS the_original_mr_fuzzy,
  CASE WHEN website_pageviews.pageview_url = '/cart' THEN 1 ELSE 0 END AS cart_flag,
  CASE WHEN website_pageviews.pageview_url = '/shipping' THEN 1 ELSE 0 END AS shipping_flag,
  CASE WHEN website_pageviews.pageview_url = '/billing' THEN 1 ELSE 0 END AS billing_flag,
  CASE WHEN website_pageviews.pageview_url = '/thank-you-for-your-order' THEN 1 ELSE 0 END AS thankyou_flag
FROM 
  bronze.website_sessions
left join bronze.website_pageviews
on website_sessions.website_session_id = website_pageviews.website_session_id
where
website_sessions.created_at > '2012-08-05' and
website_sessions.created_at < '2012-09-05' and
website_sessions.utm_source = 'gsearch' and
website_sessions.utm_campaign = 'nonbrand'
)
GROUP BY website_session_id
)
SELECT 
COUNT(DISTINCT website_session_id) as sessions,
COUNT(DISTINCT CASE WHEN lander1_made_it=1 THEN website_session_id ELSE NULL END) AS to_landingpage,
COUNT(DISTINCT CASE WHEN product_made_it=1 THEN website_session_id ELSE NULL END) AS to_product,
COUNT(DISTINCT CASE WHEN the_original_mr_fuzzy_made_it=1 THEN website_session_id ELSE NULL END) AS to_original_mr_fuzzy,
COUNT(DISTINCT CASE WHEN cart_made_it=1 THEN website_session_id ELSE NULL END) AS to_cart,
COUNT(DISTINCT CASE WHEN shipping_made_it=1 THEN website_session_id ELSE NULL END) AS to_shipping,
COUNT(DISTINCT CASE WHEN billing_made_it=1 THEN website_session_id ELSE NULL END) AS to_billing,
COUNT(DISTINCT CASE WHEN thankyou_made_it=1 THEN website_session_id ELSE NULL END) AS to_thank_you_page
FROM sessions_with_flags

In [0]:
%sql
WITH sessions_with_flags AS (
SELECT 
  website_session_id,
  max(lander1_flag) as lander1_made_it,
  max(product_flag) as product_made_it,
  max(the_original_mr_fuzzy) as the_original_mr_fuzzy_made_it,
  max(cart_flag) as cart_made_it,
  max(shipping_flag) as shipping_made_it,
  max(billing_flag) as billing_made_it,
  max(thankyou_flag) as thankyou_made_it
FROM (
SELECT 
  website_sessions.website_session_id,
  website_pageviews.website_pageview_id,
  website_pageviews.pageview_url,
  CASE WHEN website_pageviews.pageview_url = '/lander-1' THEN 1 ELSE 0 END AS lander1_flag,
  CASE WHEN website_pageviews.pageview_url = '/products' THEN 1 ELSE 0 END AS product_flag,
  CASE WHEN website_pageviews.pageview_url = '/the-original-mr-fuzzy' THEN 1 ELSE 0 END AS the_original_mr_fuzzy,
  CASE WHEN website_pageviews.pageview_url = '/cart' THEN 1 ELSE 0 END AS cart_flag,
  CASE WHEN website_pageviews.pageview_url = '/shipping' THEN 1 ELSE 0 END AS shipping_flag,
  CASE WHEN website_pageviews.pageview_url = '/billing' THEN 1 ELSE 0 END AS billing_flag,
  CASE WHEN website_pageviews.pageview_url = '/thank-you-for-your-order' THEN 1 ELSE 0 END AS thankyou_flag
FROM 
  bronze.website_sessions
left join bronze.website_pageviews
on website_sessions.website_session_id = website_pageviews.website_session_id
where
website_sessions.created_at > '2012-08-05' and
website_sessions.created_at < '2012-09-05' and
website_sessions.utm_source = 'bsearch' and
website_sessions.utm_campaign = 'nonbrand'
)
GROUP BY website_session_id
)
SELECT 
COUNT(DISTINCT website_session_id) as sessions,
concat(cast(round(100 * COUNT(DISTINCT CASE WHEN product_made_it=1 THEN website_session_id ELSE NULL END)/COUNT(DISTINCT website_session_id), 2) AS STRING), '%') as clicked_to_product,
concat(cast(round(100 * COUNT(DISTINCT CASE WHEN the_original_mr_fuzzy_made_it=1 THEN website_session_id ELSE NULL END)/COUNT(DISTINCT CASE WHEN product_made_it=1 THEN website_session_id ELSE NULL END), 2) AS STRING), '%') as clicked_to_the_original_mr_fuzzy,
concat(cast(round(100 * COUNT(DISTINCT CASE WHEN cart_made_it=1 THEN website_session_id ELSE NULL END)/COUNT(DISTINCT CASE WHEN the_original_mr_fuzzy_made_it=1 THEN website_session_id ELSE NULL END), 2) AS STRING), '%') as clieked_to_cart,
concat(cast(round(100 * COUNT(DISTINCT CASE WHEN shipping_made_it=1 THEN website_session_id ELSE NULL END)/COUNT(DISTINCT CASE WHEN cart_made_it=1 THEN website_session_id ELSE NULL END), 2) AS STRING), '%') as clicked_to_shipping,
concat(cast(round(100 * COUNT(DISTINCT CASE WHEN billing_made_it=1 THEN website_session_id ELSE NULL END)/COUNT(DISTINCT CASE WHEN shipping_made_it=1 THEN website_session_id ELSE NULL END), 2) AS STRING), '%') as clicked_to_billing,
concat(cast(round(100 * COUNT(DISTINCT CASE WHEN thankyou_made_it=1 THEN website_session_id ELSE NULL END)/COUNT(DISTINCT CASE WHEN billing_made_it=1 THEN website_session_id ELSE NULL END), 2) AS STRING), '%') as clicked_to_thank_you
FROM sessions_with_flags

In [0]:
%sql
SELECT 
COUNT(DISTINCT website_session_id) as sessions,
COUNT(DISTINCT CASE WHEN lander1_made_it=1 THEN website_session_id ELSE NULL END) AS to_landingpage,
COUNT(DISTINCT CASE WHEN product_made_it=1 THEN website_session_id ELSE NULL END) AS to_product,
COUNT(DISTINCT CASE WHEN originalmavenfuzzy_made_it=1 THEN website_session_id ELSE NULL END) AS to_original_mr_fuzzy,
COUNT(DISTINCT CASE WHEN cart_made_it=1 THEN website_session_id ELSE NULL END) AS to_cart,
COUNT(DISTINCT CASE WHEN shipping_made_it=1 THEN website_session_id ELSE NULL END) AS to_shipping,
COUNT(DISTINCT CASE WHEN billing_made_it=1 THEN website_session_id ELSE NULL END) AS to_billing,
COUNT(DISTINCT CASE WHEN thankyou_made_it=1 THEN website_session_id ELSE NULL END) AS to_thank_you_page
FROM sessions_with_flags